# Stats for the paper — reproducible computations

Computes every quantitative claim in **Thesis_Chapter_Foundation** that previously had **no saved output** in the repo, and writes them to `data/derived/stats_for_paper.json`.

Run top-to-bottom on the machine that holds the full data. Most cells read the derived parquet tables + `data/raw/{user,mlog}_demographics.csv`; the final section needs the raw impression log and is marked.

Embedded outputs below were generated 2026-07-15 from the derived tables. Values are annotated against what the paper states.

In [1]:
import json, numpy as np, polars as pl
from pathlib import Path

# Locate the repo root no matter where the kernel starts (repo root, notebooks/, etc.)
def _find_root(start):
    for d in [start, *start.parents]:
        if (d/"data"/"derived").exists() or (d/"src").is_dir():
            return d
    raise FileNotFoundError("Could not find repo root (no data/derived or src/ above %s). "
                            "Open this notebook from inside the master-thesis repo." % start)

ROOT = _find_root(Path.cwd().resolve())
DER, RAW = ROOT/"data"/"derived", ROOT/"data"/"raw"
print("repo root :", ROOT)
print("derived   :", DER, "(exists)" if DER.exists() else "(MISSING)")
print("raw       :", RAW, "(exists)" if RAW.exists() else "(MISSING)")
RESULTS = {}

repo root : /Users/benediktkasior/Documents/old computer/transfer/Master Arbeit/Git repo
derived   : /Users/benediktkasior/Documents/old computer/transfer/Master Arbeit/Git repo/data/derived (exists)
raw       : /Users/benediktkasior/Documents/old computer/transfer/Master Arbeit/Git repo/data/raw (exists)


## 1. Platform tenure distribution  
*Paper §1.2: median 23 mo, IQR 11–35, 5.6% registered ≤1 month.* No code previously computed this.

In [2]:
ud = pl.read_csv(RAW/"user_demographics.csv").select(["userId","registeredMonthCnt"])
s = ud["registeredMonthCnt"].drop_nulls()
RESULTS["tenure_distribution"] = {"n": s.len(), "median": float(s.median()),
    "q1": float(s.quantile(.25)), "q3": float(s.quantile(.75)),
    "pct_le_1_month": round((s<=1).sum()/s.len()*100, 2)}
print(RESULTS["tenure_distribution"])

{'n': 2085124, 'median': 23.0, 'q1': 11.0, 'q3': 35.0, 'pct_le_1_month': 5.6}


## 2. Taste breadth is a volume artefact  
*Paper §3.6: distinct creators vs clicks r=0.996; distinct categories vs clicks r=0.85.* Asserted in README but never computed.

In [3]:
t = pl.read_parquet(DER/"user_content_taste.parquet").filter(pl.col("ct_clicked")>0)
corr = lambda a,b: float(np.corrcoef(t[a].to_numpy(), t[b].to_numpy())[0,1])
RESULTS["taste_volume_correlations"] = {
    "creators_vs_clicks": round(corr("ct_click_n_creators","ct_clicked"),3),
    "categories_vs_clicks": round(corr("ct_click_n_content","ct_clicked"),3),
    "artists_vs_clicks": round(corr("ct_click_n_artists","ct_clicked"),3),
    "n_users_with_clicks": t.height}
print(RESULTS["taste_volume_correlations"])

{'creators_vs_clicks': 0.996, 'categories_vs_clicks': 0.845, 'artists_vs_clicks': 0.923, 'n_users_with_clicks': 145180}


## 3. Catalog counts  
*Paper §1.2/§3.6: 122 content categories, 9,914 topics.* Computed from `mlog_demographics.csv` gives 123 / 9,817 (anonymised catalog; close but not identical — reconcile with the paper's source).

In [4]:
ml = pl.read_csv(RAW/"mlog_demographics.csv")
RESULTS["catalog_counts"] = {
    "distinct_contentId_categories": ml["contentId"].n_unique(),
    "distinct_talkId_topics": ml["talkId"].n_unique(),
    "distinct_type_levels": ml["type"].n_unique()}
print(RESULTS["catalog_counts"])

{'distinct_contentId_categories': 123, 'distinct_talkId_topics': 9817, 'distinct_type_levels': 2}


## 4. Video format by activeness  
*Paper §3.6: active and inactive both ~40% video clicks, ~42% video mix.*

In [5]:
mt = pl.read_parquet(DER/"user_modeling_table.parquet").select(["userId","is_inactive"])
j = pl.read_parquet(DER/"user_content_taste.parquet").join(mt, on="userId", how="inner")
v = (j.group_by("is_inactive")
       .agg([pl.col("ct_video_share_click").mean().alias("video_share_of_clicks"),
             pl.col("ct_video_share_seen").mean().alias("video_share_of_impr")]).sort("is_inactive"))
RESULTS["video_format_by_activeness"] = {
    ("inactive" if r["is_inactive"] else "active"): {
        "video_share_of_clicks": round(r["video_share_of_clicks"],4),
        "video_share_of_impr": round(r["video_share_of_impr"],4)}
    for r in v.iter_rows(named=True)}
print(RESULTS["video_format_by_activeness"])

{'active': {'video_share_of_clicks': 0.4025, 'video_share_of_impr': 0.417}, 'inactive': {'video_share_of_clicks': 0.3652, 'video_share_of_impr': 0.4307}}


## 5. Two-stage decomposition  
*Paper §3.4: ~64% return, ~28% click given return, ~18% both.* Only the formula was in the repo.

In [6]:
wa = pl.read_parquet(DER/"user_window_agg.parquet").filter(pl.col("e_impr")>0)
n_early = wa.height; n_return = wa.filter(pl.col("l_impr")>0).height
p_click = 1 - json.load(open(DER/"eda_summary.json"))["inactive_rate_thresh_0.0"]
p_ret = n_return/n_early
RESULTS["two_stage_decomposition"] = {"n_early_users": n_early, "n_returning": n_return,
    "p_return": round(p_ret,4), "p_click_given_return": round(p_click,4),
    "p_retained_and_active": round(p_ret*p_click,4)}
print(RESULTS["two_stage_decomposition"])

{'n_early_users': 841670, 'n_returning': 542842, 'p_return': 0.645, 'p_click_given_return': 0.2833, 'p_retained_and_active': 0.1827}


## 6. Platform-tenure ablation  
*Paper §3.4: tenure lifts retention ROC-AUC 0.709→0.712 and activeness 0.705→0.707.* Computed live in `01_two_stage_models.ipynb`, never saved. Re-uses the exact model config from `src/data/two_stage.py`.

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
BEHAV = ["e_impr","e_clicks","e_likes","e_shares","e_comments","e_viewcomment","e_homepage",
         "e_active_days","e_avg_pos","e_click_rate","e_like_rate","e_avg_view_time"]
def auc(df, feats, target):
    df = df.with_columns([pl.col(c).fill_null(0.0) for c in feats])
    XY = lambda sp: (df.filter(pl.col("split")==sp).select(feats).to_numpy(),
                     df.filter(pl.col("split")==sp)[target].to_numpy().astype(int))
    Xtr,ytr = XY("train"); Xte,yte = XY("test")
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.07, max_depth=6,
        l2_regularization=1.0, random_state=42).fit(Xtr,ytr)
    return round(roc_auc_score(yte, clf.predict_proba(Xte)[:,1]), 4)
ret = pl.read_parquet(DER/"user_retention_table.parquet").join(ud, on="userId", how="left")
mt2 = pl.read_parquet(DER/"user_modeling_table.parquet").join(ud, on="userId", how="left")
f1 = BEHAV + ["ct_video_share_seen","ct_seen_n_content"]
RESULTS["tenure_ablation_roc_auc"] = {
  "stage1_retention": {"without_tenure": auc(ret,f1,"churned"),
                       "with_tenure": auc(ret,f1+["registeredMonthCnt"],"churned")},
  "stage2_activeness": {"without_tenure": auc(mt2,BEHAV,"is_inactive"),
                        "with_tenure": auc(mt2,BEHAV+["registeredMonthCnt"],"is_inactive")}}
print(RESULTS["tenure_ablation_roc_auc"])

{'stage1_retention': {'without_tenure': np.float64(0.7092), 'with_tenure': np.float64(0.7124)}, 'stage2_activeness': {'without_tenure': np.float64(0.7055), 'with_tenure': np.float64(0.7065)}}


## 7. Mean tenure by churn  
*Paper §3.4: returning users 25.9 mo vs churned 23.0 mo.*

In [8]:
ch = pl.read_parquet(DER/"user_retention_table.parquet").select(["userId","churned"]).join(ud,on="userId",how="left")
tv = ch.group_by("churned").agg(pl.col("registeredMonthCnt").mean().alias("mean_tenure")).sort("churned")
RESULTS["mean_tenure_by_churn_months"] = {("churned" if r["churned"] else "returned"): round(r["mean_tenure"],1)
    for r in tv.iter_rows(named=True)}
print(RESULTS["mean_tenure_by_churn_months"])

{'returned': 25.9, 'churned': 23.0}


## 8. Segmentation robustness + PCA  
*Paper §3.5: silhouette peaks 0.61 (k=2) / 0.59 (k=3); GMM ARI 0.74; HDBSCAN ~12 clusters, 12% noise; bootstrap ARI 0.98; PC1 40% variance.* Logic lived in `robustness.py` (print-only). Run on the 50k sample the script uses.

In [9]:
import warnings; warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
LOG = ["e_impr","e_clicks","e_likes","e_shares","e_comments","e_active_days","ct_click_n_content","ct_click_n_creators"]
RAWF = ["e_avg_pos","e_click_rate","e_avg_view_time","ct_video_share_seen","ct_video_share_click"]
FEATS = LOG + RAWF
df = (pl.read_parquet(DER/"user_modeling_table.parquet")
        .join(pl.read_parquet(DER/"user_content_taste.parquet"), on="userId", how="inner")
        .with_columns([pl.col(c).fill_null(0.0) for c in FEATS]))
X = df.select(FEATS).to_numpy().astype(float)
for j,f in enumerate(FEATS):
    if f in LOG: X[:,j] = np.log1p(np.clip(X[:,j],0,None))
Xs = StandardScaler().fit_transform(X)
rng = np.random.default_rng(42); S = Xs[rng.choice(len(Xs), 50000, replace=False)]
sil = {str(k): round(float(silhouette_score(S[:15000], KMeans(k,n_init=5,random_state=0).fit(S).labels_[:15000])),3) for k in range(2,7)}
k3 = KMeans(3,n_init=10,random_state=42).fit(S).labels_
gmm = GaussianMixture(3,n_init=3,random_state=42).fit(S).predict(S)
hdb = HDBSCAN(min_cluster_size=2000,min_samples=50).fit_predict(S)
boot = [KMeans(3,n_init=3,random_state=b).fit(S[rng.choice(len(S),len(S),replace=True)]).predict(S) for b in range(8)]
aris = [adjusted_rand_score(boot[i],boot[j]) for i in range(8) for j in range(i+1,8)]
pca = PCA(2).fit(Xs)
RESULTS["segmentation_robustness"] = {"sample_size": 50000, "silhouette_by_k": sil,
    "gmm3_vs_kmeans3_ari": round(float(adjusted_rand_score(k3,gmm)),3),
    "hdbscan": {"n_clusters": int(len(set(hdb))-(1 if -1 in hdb else 0)),
                "pct_noise": round(float((hdb==-1).mean())*100,1),
                "ari_vs_kmeans": round(float(adjusted_rand_score(k3,hdb)),3)},
    "kmeans3_bootstrap_mean_ari": round(float(np.mean(aris)),3),
    "kmeans3_bootstrap_min_ari": round(float(np.min(aris)),3),
    "pca_explained_variance_pct": {"PC1": round(float(pca.explained_variance_ratio_[0])*100,1),
                                   "PC2": round(float(pca.explained_variance_ratio_[1])*100,1)}}
print(json.dumps(RESULTS["segmentation_robustness"], indent=2))

{
  "sample_size": 50000,
  "silhouette_by_k": {
    "2": 0.613,
    "3": 0.59,
    "4": 0.336,
    "5": 0.346,
    "6": 0.348
  },
  "gmm3_vs_kmeans3_ari": 0.736,
  "hdbscan": {
    "n_clusters": 12,
    "pct_noise": 11.6,
    "ari_vs_kmeans": 0.031
  },
  "kmeans3_bootstrap_mean_ari": 0.978,
  "kmeans3_bootstrap_min_ari": 0.954,
  "pca_explained_variance_pct": {
    "PC1": 40.2,
    "PC2": 11.6
  }
}


## 9. Early-sequence length  
*Paper §4.1: median 6 early events; 50 events covers the 90th percentile.* **Correction:** counting all typed events the median is **7** (mean 13.3); ~7.4% of users hit the 50-event cap, so 50 sits near the 92nd percentile. The stated "6" matches only if events with an unknown content category are dropped.

In [10]:
z = np.load(DER/"seq_arrays.npz", allow_pickle=True)
L = (z["Xt"] > 0).sum(1)   # every impression has an image/video type; padding steps are 0
RESULTS["sequence_length_early_window"] = {"n_users": int(len(L)), "median_events": int(np.median(L)),
    "mean_events": round(float(L.mean()),2), "p90": float(np.quantile(L,.9)),
    "p95": float(np.quantile(L,.95)), "pct_at_cap_50": round(float((L>=50).mean())*100,2)}
print(RESULTS["sequence_length_early_window"])

{'n_users': 542842, 'median_events': 7, 'mean_events': 13.29, 'p90': 39.0, 'p95': 50.0, 'pct_at_cap_50': 7.43}


## 10. Needs the raw impression log (run on the Mac)

Two paper statistics cannot be produced from the derived tables — they require streaming `impression_data.csv`:

- **CTR by creator type, 3.5–5.9%** (§3.6). Needs the raw log **and** a creator-type field, which is not in the extracted demographics tables. No existing script computes it; new code + the creators table are required.
- **Taste-composition specialisation** (§3.6): 22% mean favourite-category share, ~20 categories cover 80%, composition silhouette <0.18. Produced by `src/member_a_segmentation/taste_analysis.py`, which reads the cleaned impression stream on stdin. After the patch it writes `data/derived/member_a_taste.json`:

```bash
bash src/data/run_taste_analysis.sh "../Dataset/Raw_Data.zip"
```

In [11]:
RESULTS["raw_data_required"] = {
  "ctr_by_creator_type_pct": "paper 3.5-5.9%; needs raw impression log + creator-type field. Not computed here.",
  "taste_composition_specialisation": "paper: 22% favourite-category share, ~20 cats cover 80%, silhouette <0.18. Run taste_analysis.py on full data."}

## 11. Save

In [12]:
RESULTS["_meta"] = {"generated": "2026-07-15", "source": "04_stats_for_paper.ipynb"}
json.dump(RESULTS, open(DER/"stats_for_paper.json","w"), indent=2)
print("wrote", DER/"stats_for_paper.json")

wrote /Users/benediktkasior/Documents/old computer/transfer/Master Arbeit/Git repo/data/derived/stats_for_paper.json
